# Train neural net models to predict MoA pre- and post-QC

## Import libraries

In [ ]:
import pathlib
from typing import Dict, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
from pycytominer.cyto_utils import infer_cp_features
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MultiLabelBinarizer

## Define helper functions

In [2]:
def shuffle_features(X: pd.DataFrame, random_state: int = 0) -> pd.DataFrame:
    """
    This function shuffles the values within each column of the input DataFrame
    independently, while keeping the overall structure of the DataFrame intact.
    This is useful for creating a "shuffled" version of the dataset to serve as a
    negative control in machine learning experiments.

    Args:
        X (pd.DataFrame): The input DataFrame whose features (columns) are to be
            shuffled.
        random_state (int, optional): The seed for the random number generator.
            Defaults to 0.

    Returns:
        pd.DataFrame: A new DataFrame with the same structure as the input, but with
        each column's values shuffled independently.
    """
    rng = np.random.default_rng(random_state)
    X_shuffled = X.copy()

    for col in X.columns:
        X_shuffled[col] = rng.permutation(X_shuffled[col].values)

    return X_shuffled


def split_moas(series: pd.Series) -> pd.Series:
    """Split MOA strings into lists of stripped MOA components.

    Args:
        series (pd.Series): A pandas Series containing MOA strings.
        This will be the column "Metadata_moa" in the cell painting profiles.

    Returns:
        pd.Series: A pandas Series where each element is a list of stripped
        MOA components.
    """
    return series.fillna("").apply(
        lambda x: [m.strip() for m in x.split("|") if m.strip()]
    )


def run_cv(
    X: pd.DataFrame, y: np.ndarray, hidden_layer_sizes: tuple, n_splits: int = 5
) -> float:
    """
    Perform cross-validation for a neural network model.

    Args:
        X (pd.DataFrame): The input features.
        y (np.ndarray): The target labels.
        hidden_layer_sizes (tuple): The sizes of the hidden layers in the MLP.
        n_splits (int, optional): The number of folds for cross-validation.
        Defaults to 5.

    Returns:
        float: The mean F1 score across all folds.
    """
    # Initialize KFold with the specified number of splits, shuffling, and random state
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)

    # Initialize a list to store the F1 scores for each fold
    scores = []

    # Loop through each fold generated by KFold
    for train_idx, val_idx in kf.split(X):
        # Split the data into training and validation sets based on
        # the current fold's indices
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Initialize the MLPClassifier with the specified hidden layer sizes,
        # maximum iterations, and random state
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            max_iter=200,
            random_state=0,
        )

        # Fit the model on the training data and predict on the validation set
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        # Calculate the F1 score for the current fold and append it to the scores list
        scores.append(f1_score(y_val, preds, average="micro"))

    # Return the mean F1 score across all folds
    return np.mean(scores)


def run_search(
    X: pd.DataFrame, y: np.ndarray, name: str, param_grid: list
) -> Dict[tuple, float]:
    """
    Run a search over a grid of hyperparameters for a neural network model, using
    cross-validation to evaluate each set of hyperparameters.

    Args:
        X (pd.DataFrame): The input features.
        y (np.ndarray): The target labels.
        name (str): The name of the model or experiment.
        param_grid (list): A list of tuples, each representing a set of hyperparameters.

    Returns:
        Dict[tuple, float]: A dictionary where the keys are tuples representing
        hyperparameter sets and the values are the corresponding mean F1 scores.
    """
    # Initialize a dictionary to store the results of the search
    results = {}

    # Loop through each set of hyperparameters in the parameter grid
    for h in param_grid:
        # Run cross-validation for the current set of hyperparameters
        # and store the score
        score = run_cv(X, y, h)
        results[h] = score
        # Print the results for the current set of hyperparameters
        print(f"{name} | {h} -> " f"{score:.4f}")
    return results


def train_final(
    X: pd.DataFrame,
    y: np.ndarray,
    hidden_layer_sizes: Tuple[int, ...],
    model_path: Optional[str] = None,
) -> MLPClassifier:
    """
    Train a final MLP classifier on multi-label data.

    Args:
        X (pd.DataFrame): Feature matrix.
        y (np.ndarray): Multi-label binarized target matrix.
        hidden_layer_sizes (Tuple[int, ...]): Hidden layer architecture for the MLP.
        model_path (Optional[str], optional): If provided, saves the trained model to
        this path using joblib.

    Returns:
        MLPClassifier: Trained model instance.
    """
    # Initialize the MLPClassifier with the specified hidden layer sizes, maximum
    # iterations, and random state
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes, max_iter=500, random_state=0
    )

    # Fit the model on the entire dataset
    model.fit(X, y)

    # If a model path is provided, save the trained model using joblib
    if model_path is not None:
        joblib.dump(model, model_path)

    # Return the trained model instance
    return model

## Set paths and directories

In [3]:
# Output file info
output_dir = pathlib.Path("./neural_net_models")
output_dir.mkdir(parents=True, exist_ok=True)

# Figure output directory
figure_output_dir = pathlib.Path("./figures")
figure_output_dir.mkdir(parents=True, exist_ok=True)

# Input path for single-cell profiles
input_dir = pathlib.Path(
    "/home/jenna/mnt/bandicoot/LINCS_data/processed_profiles/single_cell_profiles"
)

## Load in already created training and testing splits from LINCS paper

Link to files with training and testing splits **here**.

In [4]:
# Load in train and testing splits from original paper
train_df = pd.read_csv("/media/18tbdrive/train_lvl4_data_targets.csv.gz")
test_df = pd.read_csv("/media/18tbdrive/test_lvl4_data_targets.csv.gz")

# Print out shapes
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (27485, 1555)
Test shape: (10788, 1555)


In [5]:
# How many unique compounds in train and test?
train_df["pert_iname"].nunique(), test_df["pert_iname"].nunique()

(941, 317)

In [6]:
# Load in pre-QC cell painting profile
pre_qc_file = pathlib.Path(input_dir, "whole_batch_pre_qc_cpd_replicates.parquet")
pre_qc_df = pd.read_parquet(pre_qc_file)
# pre_qc_df = pre_qc_df.dropna(subset=["Metadata_moa"])
print(f"Pre-QC profile shape: {pre_qc_df.shape}")

# Load in post-QC cell painting profile
post_qc_file = pathlib.Path(input_dir, "whole_batch_post_qc_cpd_replicates.parquet")
post_qc_df = pd.read_parquet(post_qc_file)
# post_qc_df = post_qc_df.dropna(subset=["Metadata_moa"])
print(f"Post-QC profile shape: {post_qc_df.shape}")

Pre-QC profile shape: (51833, 807)
Post-QC profile shape: (51830, 869)


In [7]:
# How many unique compounds in pre-QC and post-QC profiles?
post_qc_df["pert_iname"].nunique(), pre_qc_df["pert_iname"].nunique()

(1553, 1553)

In [8]:
# Filter the post-QC and pre-QC profiles to only include compounds
# in the train and test sets
train_cpds = set(train_df["pert_iname"].unique())
test_cpds = set(test_df["pert_iname"].unique())

# Optional sanity check: ensure no overlap between train and test
overlap = train_cpds.intersection(test_cpds)
if overlap:
    raise ValueError(f"Train and test sets overlap on compounds: {overlap}")

# Filter both QC datasets to only include compounds present in train or test
allowed_cpds = train_cpds.union(test_cpds)

pre_qc_df = pre_qc_df[pre_qc_df["pert_iname"].isin(allowed_cpds)].reset_index(drop=True)

post_qc_df = post_qc_df[post_qc_df["pert_iname"].isin(allowed_cpds)].reset_index(
    drop=True
)

# ---- Split into train and test for each QC ----

pre_qc_train_df = pre_qc_df[pre_qc_df["pert_iname"].isin(train_cpds)].reset_index(
    drop=True
)

pre_qc_test_df = pre_qc_df[pre_qc_df["pert_iname"].isin(test_cpds)].reset_index(
    drop=True
)

post_qc_train_df = post_qc_df[post_qc_df["pert_iname"].isin(train_cpds)].reset_index(
    drop=True
)

post_qc_test_df = post_qc_df[post_qc_df["pert_iname"].isin(test_cpds)].reset_index(
    drop=True
)

# # ---- Additional sanity checks ----

print(f"Pre-QC train shape: {pre_qc_train_df.shape}")
print(f"Pre-QC test shape: {pre_qc_test_df.shape}")
print(f"Post-QC train shape: {post_qc_train_df.shape}")
print(f"Post-QC test shape: {post_qc_test_df.shape}")

# Sanity check: check the number of pert_inames in each split
print(f"Pre-QC train unique compounds: {pre_qc_train_df['pert_iname'].nunique()}")
print(f"Pre-QC test unique compounds: {pre_qc_test_df['pert_iname'].nunique()}")
print(f"Post-QC train unique compounds: {post_qc_train_df['pert_iname'].nunique()}")
print(f"Post-QC test unique compounds: {post_qc_test_df['pert_iname'].nunique()}")


# Ensure compounds are correctly separated
assert set(pre_qc_train_df["pert_iname"]).isdisjoint(pre_qc_test_df["pert_iname"])
assert set(post_qc_train_df["pert_iname"]).isdisjoint(post_qc_test_df["pert_iname"])

Pre-QC train shape: (27485, 807)
Pre-QC test shape: (10788, 807)
Post-QC train shape: (27483, 869)
Post-QC test shape: (10788, 869)
Pre-QC train unique compounds: 941
Pre-QC test unique compounds: 317
Post-QC train unique compounds: 941
Post-QC test unique compounds: 317


In [9]:
# Remove moa, broad_id, and replicate_name columns from all test/train dataframes
# and add Metadata prefix to pert_iname column
for df in [pre_qc_train_df, pre_qc_test_df, post_qc_train_df, post_qc_test_df]:
    df.drop(
        columns=["moa", "broad_id", "replicate_name"],
        inplace=True,
    )
    df.rename(columns={"pert_iname": "Metadata_pert_iname"}, inplace=True)

# Validate that the train/test splits are correct after all filtering and renaming
print(f"Pre-QC train shape: {pre_qc_train_df.shape}")
print(f"Pre-QC test shape: {pre_qc_test_df.shape}")
print(f"Post-QC train shape: {post_qc_train_df.shape}")
print(f"Post-QC test shape: {post_qc_test_df.shape}")

Pre-QC train shape: (27485, 804)
Pre-QC test shape: (10788, 804)
Post-QC train shape: (27483, 866)
Post-QC test shape: (10788, 866)


In [10]:
# Create shuffled training dfs for both pre-QC and post-QC
pre_qc_train_shuffled_df = shuffle_features(
    pre_qc_train_df.drop(columns=["Metadata_pert_iname"])
)
post_qc_train_shuffled_df = shuffle_features(
    post_qc_train_df.drop(columns=["Metadata_pert_iname"])
)

In [11]:
# Split MOAs into lists for multi-label binarization later
train_df = train_df.assign(moa_list=split_moas(train_df["Metadata_moa"]))

pre_qc_train_df = pre_qc_train_df.assign(
    moa_list=split_moas(pre_qc_train_df["Metadata_moa"])
)

pre_qc_train_shuffled_df = pre_qc_train_shuffled_df.assign(
    moa_list=split_moas(pre_qc_train_shuffled_df["Metadata_moa"])
)

post_qc_train_df = post_qc_train_df.assign(
    moa_list=split_moas(post_qc_train_df["Metadata_moa"])
)

post_qc_train_shuffled_df = post_qc_train_shuffled_df.assign(
    moa_list=split_moas(post_qc_train_shuffled_df["Metadata_moa"])
)

/tmp/ipykernel_2726678/4180273338.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df = train_df.assign(moa_list=split_moas(train_df["Metadata_moa"]))
/tmp/ipykernel_2726678/4180273338.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pre_qc_train_shuffled_df = pre_qc_train_shuffled_df.assign(
/tmp/ipykernel_2726678/4180273338.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usi

In [12]:
# Set up multi-label binarizer and fit on the training MOAs
mlb = MultiLabelBinarizer()

# Fit the multi-label binarizer on the training MOAs (using the original train_df)
mlb.fit(train_df["moa_list"])

# Transform the MOA lists into binary indicator arrays for each training set
y_pre_train = mlb.transform(pre_qc_train_df["moa_list"])
y_pre_train_shuffled = mlb.transform(pre_qc_train_shuffled_df["moa_list"])

y_post_train = mlb.transform(post_qc_train_df["moa_list"])
y_post_train_shuffled = mlb.transform(post_qc_train_shuffled_df["moa_list"])

# Sanity check: print the shape of the resulting binary indicator arrays
print("Pre final:", y_pre_train.shape)
print("Pre shuf:", y_pre_train_shuffled.shape)

print("Post final:", y_post_train.shape)
print("Post shuf:", y_post_train_shuffled.shape)

print("MOA classes:", len(mlb.classes_))

Pre final: (27485, 439)
Pre shuf: (27485, 439)
Post final: (27483, 439)
Post shuf: (27483, 439)
MOA classes: 439


In [13]:
# Set metadata cols and feature cols
pre_metadata_cols = infer_cp_features(pre_qc_train_df, metadata=True)
pre_feature_cols = infer_cp_features(pre_qc_train_df, metadata=False)
post_metadata_cols = infer_cp_features(post_qc_train_df, metadata=True)
post_feature_cols = infer_cp_features(post_qc_train_df, metadata=False)

# Create X for pre and post QC dfs
X_pre_real = pre_qc_train_df[pre_feature_cols]
X_pre_shuffled = pre_qc_train_shuffled_df[pre_feature_cols]
X_post_real = post_qc_train_df[post_feature_cols]
X_post_shuffled = post_qc_train_shuffled_df[post_feature_cols]

In [14]:
# Define the parameter grid for hidden layer sizes to test
param_grid = [
    (64,),
    (128,),
    (128, 64),
]

# Run the search for each dataset and store results in a dictionary
pre_qc_final_results = run_search(X_pre_real, y_pre_train, "pre_qc_final", param_grid)
pre_qc_shuf_results = run_search(
    X_pre_shuffled, y_pre_train_shuffled, "pre_qc_shuf", param_grid
)

post_qc_final_results = run_search(
    X_post_real, y_post_train, "post_qc_final", param_grid
)
post_qc_shuf_results = run_search(
    X_post_shuffled, y_post_train_shuffled, "post_qc_shuf", param_grid
)

/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


pre_qc_final | (64,) -> 0.2614
pre_qc_final | (128,) -> 0.3095
pre_qc_final | (128, 64) -> 0.2804


/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: Con

pre_qc_shuf | (64,) -> 0.0037
pre_qc_shuf | (128,) -> 0.0087
pre_qc_shuf | (128, 64) -> 0.0089


/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


post_qc_final | (64,) -> 0.2296
post_qc_final | (128,) -> 0.2832
post_qc_final | (128, 64) -> 0.2484


/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: Con

post_qc_shuf | (64,) -> 0.0040
post_qc_shuf | (128,) -> 0.0089
post_qc_shuf | (128, 64) -> 0.0089


In [15]:
# Identify the best hyperparameters for each dataset based on the highest F1 score
best_pre_qc_final = max(pre_qc_final_results, key=pre_qc_final_results.get)
best_pre_qc_shuf = max(pre_qc_shuf_results, key=pre_qc_shuf_results.get)
best_post_qc_final = max(post_qc_final_results, key=post_qc_final_results.get)
best_post_qc_shuf = max(post_qc_shuf_results, key=post_qc_shuf_results.get)

In [16]:
# Ensure model directory exists
model_dir = pathlib.Path("models")
model_dir.mkdir(exist_ok=True)

# Define all experiments
experiments = {
    "pre_qc_final": (X_pre_real, y_pre_train, best_pre_qc_final),
    "pre_qc_shuffled": (X_pre_shuffled, y_pre_train_shuffled, best_pre_qc_shuf),
    "post_qc_final": (X_post_real, y_post_train, best_post_qc_final),
    "post_qc_shuffled": (X_post_shuffled, y_post_train_shuffled, best_post_qc_shuf),
}

# Train + save loop
trained_models = {}
for name, (X, y, params) in experiments.items():
    print(f"Training {name}...")

    model = train_final(
        X=X,
        y=y,
        hidden_layer_sizes=params,
        model_path=str(model_dir / f"{name}_mlp.joblib"),
    )

    trained_models[name] = model

print("Done training all models.")

Training pre_qc_final...
Training pre_qc_shuffled...
Training post_qc_final...
Training post_qc_shuffled...
Done training all models.
